## Collaborative Recommendation system

In [1]:
import torch
from torch import nn
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn import preprocessing, model_selection
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv("datasets/ratings.csv")
df.head(5)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
le_users = preprocessing.LabelEncoder()
le_movies = preprocessing.LabelEncoder()
df.userId = le_users.fit_transform(df.userId.values)
df.movieId = le_movies.fit_transform(df.movieId.values)

In [4]:
df.head(5)

,userId,movieId,rating,timestamp
0,0,0,4.0,964982703
1,0,2,4.0,964981247
2,0,5,4.0,964982224
3,0,43,5.0,964983815
4,0,46,5.0,964982931


In [5]:
len(le_users.classes_), len(le_movies.classes_)

(610, 9724)

In [6]:
df_train, df_test = model_selection.train_test_split(df, test_size=0.2, random_state=42, stratify=df.rating.values)

In [7]:
class MoviesData(Dataset):
    def __init__(self, users, movies, ratings):
        super().__init__()
        self.users = torch.tensor(users, dtype=torch.long)
        self.movies = torch.tensor(movies, dtype=torch.long)
        self.ratings = torch.tensor(ratings, dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]        

In [8]:
train_dataset = MoviesData(users=df_train.userId.values, movies=df_train.movieId.values, ratings=df_train.rating.values)
test_dataset = MoviesData(users=df_test.userId.values, movies=df_test.movieId.values, ratings=df_test.rating.values)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
class RSModel(nn.Module):
    def __init__(self, n_users, n_movies, n_embeddings=32):
        super().__init__()
        self.n_users = n_users
        self.n_movies = n_movies
        self.n_embeddings = n_embeddings
        self.user_embeddings = nn.Embedding(self.n_users, self.n_embeddings)
        self.movie_embeddings = nn.Embedding(self.n_movies, self.n_embeddings)
        self.fc = nn.Linear(self.n_embeddings*2, 1)

    def forward(self, users, movies):
        user_embedd = self.user_embeddings(users)
        movie_embedds = self.movie_embeddings(movies)
        outputs = torch.cat([user_embedd, movie_embedds], dim=1)
        output = self.fc(outputs)
        return output

In [11]:
model = RSModel(len(le_users.classes_), len(le_movies.classes_))
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [12]:
epochs = 1

for i in range(epochs):
    model.train()    
    for users, movies, ratings in train_loader:
        optimizer.zero_grad()
        y_pred_ratings = model(users, movies)
        y_true = ratings.unsqueeze(dim=1).to(torch.float32)
        loss = loss_fn(y_pred_ratings, y_true)
        loss.backward()
        optimizer.step()

In [13]:
y_preds = []
y_trues = []

model.eval()
with torch.no_grad():
    for users, movies, ratings in test_loader: 
        y_true = ratings.detach().numpy().tolist()
        y_pred = model(users, movies).squeeze().detach().numpy().tolist()
        y_trues.append(y_true)
        y_preds.append(y_pred)

In [14]:
filtered = [(t, p) for t, p in zip(y_trues, y_preds)
            if len(t) == 32 and len(p) == 32]

y_trues_fixed = np.array([t for t, p in filtered])
y_preds_fixed = np.array([p for t, p in filtered])

mse = mean_squared_error(y_trues_fixed, y_preds_fixed)
print("MSE:", mse)

MSE: 1.1530521707256471
